In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ===============================================================
# Store Sales - Time Series Forecasting (Clean, Fixed & Working)
# ===============================================================

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import warnings, os

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
DATA_DIR = "/kaggle/input/store-sales-time-series-forecasting"
VAL_DAYS = 16
SEED = 42

# ---------------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------------
train = pd.read_csv(f"{DATA_DIR}/train.csv", parse_dates=['date'])
test = pd.read_csv(f"{DATA_DIR}/test.csv", parse_dates=['date'])
stores = pd.read_csv(f"{DATA_DIR}/stores.csv")
oil = pd.read_csv(f"{DATA_DIR}/oil.csv", parse_dates=['date'])
holidays = pd.read_csv(f"{DATA_DIR}/holidays_events.csv", parse_dates=['date'])
transactions = pd.read_csv(f"{DATA_DIR}/transactions.csv", parse_dates=['date'])
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

# ---------------------------------------------------------------
# PREPROCESSING
# ---------------------------------------------------------------
# Fill missing oil prices
oil['dcoilwtico'] = oil['dcoilwtico'].ffill().bfill()

# Merge base data
for df in [train, test]:
    df.merge(stores, on="store_nbr", how="left")
    df.merge(oil, on="date", how="left")

# Holidays simplified
holidays['is_holiday'] = holidays['type'].isin(['Holiday', 'Additional']).astype(int)
holidays = holidays[['date', 'is_holiday']]

train = train.merge(stores, on="store_nbr", how="left")
test = test.merge(stores, on="store_nbr", how="left")

train = train.merge(oil, on="date", how="left")
test = test.merge(oil, on="date", how="left")

train = train.merge(holidays, on="date", how="left")
test = test.merge(holidays, on="date", how="left")

train['is_holiday'] = train['is_holiday'].fillna(0)
test['is_holiday'] = test['is_holiday'].fillna(0)

train = train.merge(transactions, on=["date", "store_nbr"], how="left")
test = test.merge(transactions, on=["date", "store_nbr"], how="left")

train['transactions'] = train['transactions'].fillna(0)
test['transactions'] = test['transactions'].fillna(0)

# ---------------------------------------------------------------
# FEATURE ENGINEERING
# ---------------------------------------------------------------
for df in [train, test]:
    df['day'] = df['date'].dt.day
    df['dow'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    df['is_weekend'] = df['dow'].isin([5, 6]).astype(int)

# Label encode categorical columns
for col in ['family', 'city', 'state', 'type']:
    le = LabelEncoder()
    le.fit(pd.concat([train[col], test[col]], axis=0).astype(str))
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

# Create unique id for lag features
train['id'] = train['store_nbr'].astype(str) + "_" + train['family'].astype(str)
test['id'] = test['store_nbr'].astype(str) + "_" + test['family'].astype(str)

# Sort
train = train.sort_values(['id', 'date'])
test = test.sort_values(['id', 'date'])

# ---------------------------------------------------------------
# CREATE LAGS FOR BOTH TRAIN AND TEST
# ---------------------------------------------------------------
full_df = pd.concat([train[['id', 'date', 'sales']], test[['id', 'date']]], axis=0, sort=False)
full_df = full_df.sort_values(['id', 'date'])

for lag in [7, 14, 28]:
    full_df[f'lag_{lag}'] = full_df.groupby('id')['sales'].shift(lag)
    full_df[f'roll_mean_{lag}'] = (
        full_df.groupby('id')['sales']
        .shift(7)
        .rolling(lag)
        .mean()
        .reset_index(0, drop=True)
    )

# Merge lag features back
train = train.merge(full_df[['id', 'date', 'lag_7', 'lag_14', 'lag_28',
                             'roll_mean_7', 'roll_mean_14', 'roll_mean_28']],
                    on=['id', 'date'], how='left')
test = test.merge(full_df[['id', 'date', 'lag_7', 'lag_14', 'lag_28',
                           'roll_mean_7', 'roll_mean_14', 'roll_mean_28']],
                  on=['id', 'date'], how='left')

# Drop NaNs in train
train = train.dropna(subset=['lag_7', 'lag_14', 'lag_28']).reset_index(drop=True)

# ---------------------------------------------------------------
# TRAIN / VALID SPLIT
# ---------------------------------------------------------------
max_date = train['date'].max()
val_start = max_date - pd.Timedelta(days=VAL_DAYS - 1)
train_mask = train['date'] < val_start
val_mask = train['date'] >= val_start

features = [
    'onpromotion', 'transactions', 'dcoilwtico', 'is_holiday',
    'day', 'dow', 'month', 'year', 'is_weekend',
    'family', 'city', 'state', 'type',
    'lag_7', 'lag_14', 'lag_28',
    'roll_mean_7', 'roll_mean_14', 'roll_mean_28'
]

X_train, y_train = train.loc[train_mask, features], np.log1p(train.loc[train_mask, 'sales'])
X_val, y_val = train.loc[val_mask, features], np.log1p(train.loc[val_mask, 'sales'])

# ---------------------------------------------------------------
# LIGHTGBM MODEL
# ---------------------------------------------------------------
model = lgb.LGBMRegressor(
    objective="regression",
    learning_rate=0.05,
    num_leaves=128,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    random_state=SEED,
    n_estimators=5000,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="rmse",
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)]
)

# ---------------------------------------------------------------
# VALIDATION METRIC
# ---------------------------------------------------------------
val_pred = np.expm1(model.predict(X_val))
y_true = np.expm1(y_val)
rmsle = np.sqrt(np.mean(np.square(np.log1p(val_pred) - np.log1p(y_true))))
print(f"✅ Validation RMSLE: {rmsle:.5f}")

# ---------------------------------------------------------------
# TEST PREDICTION
# ---------------------------------------------------------------
X_test = test[features].fillna(0)
test_pred = np.expm1(model.predict(X_test))
test_pred = np.clip(test_pred, 0, None)

# ---------------------------------------------------------------
# SUBMISSION
# ---------------------------------------------------------------
submission = sample_sub.copy()
submission['sales'] = test_pred
submission.to_csv("submission.csv", index=False)
print("✅ submission.csv saved successfully!")

# ---------------------------------------------------------------
# FEATURE IMPORTANCE PLOT
# ---------------------------------------------------------------
lgb.plot_importance(model, importance_type='gain', max_num_features=15, figsize=(8, 6))
plt.title("Top 15 Feature Importances")
plt.tight_layout()
plt.savefig("feature_importance.png")
plt.show()
